# Clinical Symptom Validation and Multi-Source Lookup

This notebook uses `CentralKnowledgeLookup` to validate clinical concepts (symptoms, signs, findings and diseases) against UMLS, BioPortal and OLS. UMLS **semantic types** decide whether a concept is clinically relevant.

Requirements: `pip install "biomedical-knowledge-lookup[umls]"`, plus a `UMLS_API_KEY` (or `UMLS_API_KEY_TU`) and a `BIOPORTAL_API_KEY` in your environment or in a `.env` file.

## 1. Setup environment and API keys

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

from knowledge_lookup import CentralKnowledgeLookup, KnowledgeSource, LookupConfig

# Search upward from the notebook's working directory for a .env file
load_dotenv(find_dotenv(usecwd=True))

api_keys = {
    "umls": os.environ.get("UMLS_API_KEY") or os.environ.get("UMLS_API_KEY_TU"),
    "bioportal": os.environ.get("BIOPORTAL_API_KEY"),
}
missing = [name for name, key in api_keys.items() if not key]
if missing:
    print(f"Missing API keys for: {missing} - those sources will return no results.")

config = LookupConfig(
    api_keys={name: key for name, key in api_keys.items() if key},
    enabled_sources=[KnowledgeSource.UMLS, KnowledgeSource.BIOPORTAL, KnowledgeSource.OLS],
)

lookup = CentralKnowledgeLookup(config)
print("Central lookup system initialized.")

## 2. Validating an existing CUI

Datasets often contain CUIs (Concept Unique Identifiers) that point to unrelated or non-clinical concepts. We fetch the UMLS record and check its semantic types against a set of clinical types.

`UnifiedConcept.semantic_types` holds semantic type *names* such as `"Sign or Symptom"`, so the table below maps the familiar TUIs to those names.

In [ ]:
# Clinical UMLS semantic types (TUI -> name)
CLINICAL_SEMANTIC_TYPES = {
    "T047": "Disease or Syndrome",
    "T184": "Sign or Symptom",
    "T033": "Finding",
    "T037": "Injury or Poisoning",
    "T190": "Anatomical Abnormality",
    "T048": "Mental or Behavioral Dysfunction",
}
CLINICAL_TYPE_NAMES = set(CLINICAL_SEMANTIC_TYPES.values())


def clinical_types(concept):
    """Return the clinical semantic types of a concept (empty if none)."""
    return [st for st in concept.semantic_types or [] if st in CLINICAL_TYPE_NAMES]


async def validate_clinical_cui(cui):
    print(f"Validating CUI: {cui}...")
    concept = await lookup.get_concept_details(cui, source=KnowledgeSource.UMLS)

    if not concept:
        return "CUI not found in UMLS."

    print(f"  Primary label:  {concept.primary_label}")
    print(f"  Semantic types: {concept.semantic_types}")

    if clinical_types(concept):
        return "VALID CLINICAL CONCEPT"
    return "INVALID TYPE: non-clinical concept"


# A correct CUI for headache
print(await validate_clinical_cui("C0018681"))

print("\n---\n")

# A CUI that is not a clinical finding (Aspirin, a pharmacologic substance)
print(await validate_clinical_cui("C0004057"))

## 3. Searching with unified scoring

When a term has no CUI, we search all enabled sources and score each candidate by fuzzy label similarity, source agreement and the adapter's confidence.

Search results don't carry semantic types, so UMLS candidates are enriched with `get_concept_details` before the clinical filter is applied.

In [ ]:
import asyncio
import difflib


def umls_cui(concept):
    """Return the concept's UMLS CUI, if any source contributed one."""
    for identifier in concept.identifiers or []:
        if identifier.source == KnowledgeSource.UMLS:
            return identifier.identifier
    return None


async def unified_clinical_search(term, max_results=10):
    print(f"Searching for clinical matches for: '{term}'...")
    result = await lookup.search_concepts(term, max_results=max_results)
    sources_queried = max(len(result.sources_queried or []), 1)

    candidates = [(concept, umls_cui(concept)) for concept in result.concepts or []]
    candidates = [(concept, cui) for concept, cui in candidates if cui]
    details = await asyncio.gather(
        *(lookup.get_concept_details(cui, source=KnowledgeSource.UMLS) for _, cui in candidates)
    )

    scored_matches = []
    for (concept, cui), detail in zip(candidates, details):
        # 1. Keep only clinical concepts
        if detail is None or not clinical_types(detail):
            continue

        # 2. Fuzzy match score between query and label
        fuzzy_score = difflib.SequenceMatcher(
            None, term.lower(), concept.primary_label.lower()
        ).ratio()

        # 3. Source agreement score
        source_score = len(concept.sources or []) / sources_queried

        # 4. Final combined confidence
        final_score = (
            fuzzy_score * 0.5 + source_score * 0.3 + (concept.confidence_score or 0.0) * 0.2
        )

        scored_matches.append(
            {
                "cui": cui,
                "label": concept.primary_label,
                "types": clinical_types(detail),
                "score": round(final_score, 3),
                "sources": [str(source) for source in concept.sources or []],
            }
        )

    scored_matches.sort(key=lambda match: match["score"], reverse=True)
    return scored_matches


matches = await unified_clinical_search("shortness of breath")
for m in matches:
    print(
        f"Score {m['score']}: {m['cui']} ({m['label']}) "
        f"{m['types']} [Sources: {', '.join(m['sources'])}]"
    )

## 4. Cleanup

Close the lookup system to release HTTP sessions.

In [ ]:
await lookup.close()
print("Lookup system closed.")